<a href="https://colab.research.google.com/github/Quantium/Zeus-Notebooks/blob/main/ZeusPipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title Installations
# !pip install rich #Already in colab vms
!pip install rich-tools pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 13.0 MB/s eta 0:00:00
  Attempting uninstall: rich
    Found existing installation: rich 13.9.4
    Uninstalling rich-13.9.4:
      Successfully uninstalled rich-13.9.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.40.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.


In [2]:
#@title Base Libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

In [3]:
#@title Select for Dark Mode
import seaborn as sns

darkmode_selector = True #@param {type:"boolean"}
if darkmode_selector:
  plt.style.use('seaborn-v0_8-whitegrid')

  plt.style.use('dark_background')
  plt.rcParams.update({
      'axes.edgecolor': 'white',
      'axes.labelcolor': 'white',
      'text.color': 'white',
      'xtick.color': 'white',
      'ytick.color': 'white',
      'grid.color': 'lightgray',
      'grid.linestyle': '--',
      'grid.linewidth': 0.5,
      'lines.linewidth': 1.5,
      'lines.markersize': 5,
      'font.size': 10,
      'figure.figsize': (8, 6),
      'savefig.dpi': 300,
      'savefig.bbox': 'tight',
      'legend.edgecolor': 'white',
      'legend.frameon': True,
      'legend.fancybox': False,
  })

  #Para as matrices de correlación
  dark_cmap = sns.color_palette("binary", as_cmap=True)

In [4]:
#@title ML Libraries

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate, GridSearchCV,RandomizedSearchCV
from sklearn.feature_selection import SelectKBest, chi2, VarianceThreshold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score, RocCurveDisplay,accuracy_score, roc_curve, auc, classification_report,confusion_matrix,precision_recall_curve, precision_recall_fscore_support,fbeta_score, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso, LinearRegression, HuberRegressor

from scipy.stats import kurtosis, skew

In [5]:
#@title Función para guardar objetos con joblib
## Crea o carga un objeto, ideal para objetos que toman mucho tiempo en generarse
def create_or_load(folder_path, filename, creation_function):
  file_path=os.path.join(folder_path, filename)
  if not os.path.exists(file_path):
      print(f"Joblib file not found. Creating {file_path}")
      data = creation_function()
      joblib.dump(data, file_path)

  print(f"Joblib file found.{file_path}")
  return joblib.load(file_path)

In [6]:
#@title Google Drive
from google.colab import drive
from google.colab import auth
import gspread
from google.auth import default
import pandas as pd

auth.authenticate_user()
creds, _ = default() y
gc = gspread.authorize(creds)

drive.mount('/content/drive')
working_folder = "/content/drive/MyDrive/Darovi Consulting/Zeus/reextraccindedatosproyectohades" # @param {type:"string"}

working_file = 'Columnas para Recibo OC  lineas y pedido'

Mounted at /content/drive


# Ordenes de Compra Lineas


```SQL
WITH ARTICULO_DETALLE AS (
SELECT A.CODIGO_ARTICULO,
FF.DESCRIPCION FORMA_FARMACEUTICA,
F1.DESCRIPCION D_CODIGO_FAMILIA,
F3.DESCRIPCION SUSTANCIA_ACTIVA,
F4.DESCRIPCION CONTROLADO,
F6.DESCRIPCION TEMPERATURA,
F10.DESCRIPCION CLASIFICACION,
F10.DESCRIPCION SUB_FAMILIA
FROM ARTICULOS A
LEFT JOIN fe_forma_farmaceutica_sat FF
ON FF.CLAVE=A.AUXILIAR_1
LEFT JOIN FAMILIAS F1
ON F1.CODIGO_FAMILIA=A.CODIGO_FAMILIA
AND F1.NUMERO_TABLA=1
AND F1.ULTIMO_NIVEL='S'
AND F1.CODIGO_EMPRESA=A.CODIGO_EMPRESA
LEFT JOIN FAMILIAS F3
ON F3.CODIGO_FAMILIA=A.CODIGO_ESTAD3
AND F3.NUMERO_TABLA=3
AND F3.ULTIMO_NIVEL='S'
AND F3.CODIGO_EMPRESA=A.CODIGO_EMPRESA
LEFT JOIN FAMILIAS F4
ON F4.CODIGO_FAMILIA=A.CODIGO_ESTAD4
AND F4.NUMERO_TABLA=4
AND F4.ULTIMO_NIVEL='S'
AND F4.CODIGO_EMPRESA=A.CODIGO_EMPRESA
LEFT JOIN FAMILIAS F6
ON F6.CODIGO_FAMILIA=A.CODIGO_ESTAD6
AND F6.NUMERO_TABLA=6
AND F6.ULTIMO_NIVEL='S'
AND F6.CODIGO_EMPRESA=A.CODIGO_EMPRESA
WHERE A.CODIGO_EMPRESA = '01'
LEFT JOIN FAMILIAS F11
ON F10.CODIGO_FAMILIA=A.CODIGO_ESTAD11
AND F10.NUMERO_TABLA=11
AND F10.ULTIMO_NIVEL='S'
AND F10.CODIGO_EMPRESA=A.CODIGO_EMPRESA
WHERE A.CODIGO_EMPRESA = '01'
LEFT JOIN FAMILIAS F12
ON F10.CODIGO_FAMILIA=A.CODIGO_ESTAD12
AND F10.NUMERO_TABLA=12
AND F10.ULTIMO_NIVEL='S'
AND F10.CODIGO_EMPRESA=A.CODIGO_EMPRESA
WHERE A.CODIGO_EMPRESA = '01'
)
SELECT PCL.*,
AD.*
FROM PEDIDOS_COMPRAS PC,
PEDIDOS_COMPRAS_LIN PCL,
ARTICULO_DETALLE AD
WHERE PCL.CODIGO_EMPRESA=PC.CODIGO_EMPRESA
AND PCL.ORGANIZACION_COMPRAS=PC.ORGANIZACION_COMPRAS
AND PCL.SERIE_NUMERACION=PC.SERIE_NUMERACION
AND PCL.NUMERO_PEDIDO=PC.NUMERO_PEDIDO
AND PCL.CODIGO_ARTICULO=AD.CODIGO_ARTICULO
AND PC.CODIGO_EMPRESA = '01'
--AND PC.FECHA_PEDIDO >= '05/05/2026'
```




In [21]:
#@title Líneas DataFrame
lineas = gc.open(working_file).worksheet('OC lineas') #PEDIDOS_COMPRAS_LIN

rows = lineas.get_all_values()
lineasSucios_df = pd.DataFrame.from_records(rows[1:], columns=rows[0])
lineas_df=lineasSucios_df[['DATE','ID','CODIGO_EMPRESA','ORGANIZACION_COMPRAS','SERIE_NUMERACION','NUMERO_PEDIDO','CODIGO_ARTICULO','DESCRIPCION','FECHA_ENTREGA','FECHA_ENTREGA_CONFIRM','UNIDADES_PEDIDAS','STATUS_CIERRE','STATUS_APROBADO','FORMA_FARMACEUTICA','D_CODIGO_FAMILIA']]
lineas_df.columns

Index(['DATE', 'ID', 'CODIGO_EMPRESA', 'ORGANIZACION_COMPRAS',
       'SERIE_NUMERACION', 'NUMERO_PEDIDO', 'CODIGO_ARTICULO', 'DESCRIPCION',
       'FECHA_ENTREGA', 'FECHA_ENTREGA_CONFIRM', 'UNIDADES_PEDIDAS',
       'STATUS_CIERRE', 'STATUS_APROBADO', 'FORMA_FARMACEUTICA',
       'D_CODIGO_FAMILIA'],
      dtype='object')

# Ordenes de Compra Cabecera

```SQL
SELECT PC.*
FROM PEDIDOS_COMPRAS PC
WHERE PC.CODIGO_EMPRESA='01'
--AND PC.FECHA_PEDIDO>=TO_DATE('05/05/2026','DD/MM/YYYY')
```



In [22]:
#@title Pedidos DataFrame
pedidos = gc.open(working_file).worksheet(' OC pedidos') #PEDIDOS_COMPRAS

rows = pedidos.get_all_values()
pedidosSucio_df = pd.DataFrame.from_records(rows[1:], columns=rows[0])
pedidos_df=pedidosSucio_df[['DATE','ID','CODIGO_EMPRESA','ORGANIZACION_COMPRAS','TIPO_PEDIDO','SERIE_NUMERACION','NUMERO_PEDIDO','CODIGO_PROVEEDOR','FECHA_PEDIDO','CODIGO_ALMACEN','USUARIO_ALTA','FECHA_ALTA','STATUS_CIERRE','STATUS_APROBADO']]
pedidos_df.columns

Index(['DATE', 'ID', 'CODIGO_EMPRESA', 'ORGANIZACION_COMPRAS', 'TIPO_PEDIDO',
       'SERIE_NUMERACION', 'NUMERO_PEDIDO', 'CODIGO_PROVEEDOR', 'FECHA_PEDIDO',
       'CODIGO_ALMACEN', 'USUARIO_ALTA', 'FECHA_ALTA', 'STATUS_CIERRE',
       'STATUS_APROBADO'],
      dtype='object')


```SQL
# Calendario de Citas
SELECT
    CEC.NUMERO_EXPEDIENTE,
    CL.FECHA_INICIO,
    CL.LIBREA01 HORA_INICIO,
    PC.D_CODIGO_PROVEEDOR,
    PCL.SERIE_NUMERACION,
PCL.NUMERO_PEDIDO,
PCL.CODIGO_ARTICULO,
PCL.DESCRIPCION,
PCL.PRESENTACION_PEDIDO,
    SUM(PCL.UNIDADES_PEDIDAS) CANTIDAD,
SUM(SUM(PCL.UNIDADES_PEDIDAS)) OVER (
        PARTITION BY CEC.NUMERO_EXPEDIENTE
    ) PIEZAS_TOTALES
FROM CRMEXPEDIENTES_CAB CEC
JOIN (
    SELECT
        PC.*,
        (
            SELECT PRLV.NOMBRE
            FROM PROVEEDORES PRLV
            WHERE PRLV.CODIGO_RAPIDO = PC.CODIGO_PROVEEDOR
              AND PRLV.CODIGO_EMPRESA = PC.CODIGO_EMPRESA
        ) D_CODIGO_PROVEEDOR
    FROM PEDIDOS_COMPRAS PC
) PC
    ON PC.CODIGO_EMPRESA = CEC.EMPRESA
   AND PC.RESERVADO_NUMBER1 = CEC.NUMERO_EXPEDIENTE
JOIN CALENDARIOS_LIBRA CL
    ON CL.LIBREN01 = CEC.NUMERO_EXPEDIENTE
JOIN PEDIDOS_COMPRAS_LIN PCL
    ON PCL.CODIGO_EMPRESA = PC.CODIGO_EMPRESA
   AND PCL.ORGANIZACION_COMPRAS = PC.ORGANIZACION_COMPRAS
   AND PCL.SERIE_NUMERACION = PC.SERIE_NUMERACION
   AND PCL.NUMERO_PEDIDO = PC.NUMERO_PEDIDO
WHERE CEC.TIPO_EXPEDIENTE = '70011'
  --AND CL.CODIGO_CALENDARIO = '2'
AND CL.TIPO_CALENDARIO= 'EMP'
  --AND CL.FECHA_INICIO > TO_DATE('01/04/2026', 'DD/MM/YYYY')
GROUP BY
    CEC.NUMERO_EXPEDIENTE,
    CL.FECHA_INICIO,
    CL.LIBREA01,
    PC.D_CODIGO_PROVEEDOR,
    PCL.SERIE_NUMERACION,
PCL.NUMERO_PEDIDO,
   PCL.CODIGO_ARTICULO,
    PCL.DESCRIPCION,
    PCL.PRESENTACION_PEDIDO
ORDER BY
    1, 3, 4
```

In [15]:
#@title Calendario de Citas DataFrame
citas = gc.open(working_file).worksheet('Calendario de Citas')

rows = citas.get_all_values()
citas_df = pd.DataFrame.from_records(rows[1:], columns=rows[0])
citas_df.columns

Index(['FECHA_INICIO', 'HORA_INICIO', 'D_CODIGO_PROVEEDOR', 'SERIE_NUMERACION',
       'CODIGO_ARTICULO', 'DESCRIPCION', 'PRESENTACION_PEDIDO',
       'NUMERO_EXPEDIENTE', 'NUMERO_PEDIDO', 'CANTIDAD', 'PIEZAS_TOTALES'],
      dtype='object')


```SQL
SELECT *
FROM RF_TAREAS_01
WHERE FECHA_GEN>=TO_DATE('05/05/2026','DD/MM/YYYY')
```

In [16]:
#@title Entradas Logisticas DataFrame
entradas = gc.open(working_file).worksheet('Entradas Logisticas')

rows = entradas.get_all_values()
entradas_df = pd.DataFrame.from_records(rows[1:], columns=rows[0])
entradas_df.columns

Index(['EMPRESA', 'CONTADOR', 'FECHA_GEN', 'USUARIO_GEN', 'MAQUINA_GEN',
       'FECHA_INICIO', 'FECHA_FIN', 'MAQUINA', 'USUARIO', 'OBSERVACIONES',
       'PRIORIDAD', 'STATUS', 'FECHA_DOC_INT', 'DOCUMENTO_INT',
       'LINEA_COMPRAS', 'FECHA_DOC_EXT', 'DOCUMENTO_EXT', 'ENTIDAD',
       'ARTICULO', 'LOTE_INT', 'LOTE_PROV', 'SERIE_INT', 'SERIE_PROV',
       'NUM_PALET', 'ALMACEN', 'ZONA_ORIGEN', 'SITUACION_ORIGEN',
       'UBICACION_ORIGEN', 'ZONA_DESTINO', 'SITUACION_DESTINO',
       'UBICACION_DESTINO', 'CANT_PRES', 'PRESENTACION', 'CAD_LOGISTICA',
       'UNIDADES1', 'UNIDADES2', 'ZONA_TRABAJO', 'MOTIVO_PARADA',
       'CONTADOR_ORIGEN', 'NUM_ETI_ENT', 'EJERCICIO_PEDIDO', 'SERIE_PEDIDO',
       'NUMERO_PEDIDO', 'COD_MVTO_COMPRA', 'COD_MVTO_COLOCA',
       'UNIDADES1_DEVOL', 'UNIDADES2_DEVOL', 'UBICACION_CARGA', 'CODIGO_MENU',
       'BOTON_CAMBIO_UBICACION', 'MAQUINA_ANT', 'ORGANIZACION_COMPRAS',
       'CODIGO_ORG_PLANTA', 'ORDEN_DE_FABRICACION', 'MOTIVO_BLOQUEO',
       'STATUS_ANT

```SQL
SELECT *
FROM RF_TAREAS_02
WHERE FECHA_GEN>=TO_DATE('05/05/2026','DD/MM/YYYY')
```

In [17]:
#@title Ubicaciones DataFrame
Ubicaciones = gc.open(working_file).worksheet('Ubicaciones')

rows = Ubicaciones.get_all_values()
Ubicaciones_df = pd.DataFrame.from_records(rows[1:], columns=rows[0])
Ubicaciones_df.columns

Index(['EMPRESA', 'CONTADOR', 'FECHA_GEN', 'USUARIO_GEN', 'MAQUINA_GEN',
       'FECHA_INICIO', 'FECHA_FIN', 'MAQUINA', 'USUARIO', 'OBSERVACIONES',
       'PRIORIDAD', 'STATUS', 'FECHA_DOC_INT', 'DOCUMENTO_INT',
       'LINEA_COMPRAS', 'FECHA_DOC_EXT', 'DOCUMENTO_EXT', 'ENTIDAD',
       'ARTICULO', 'LOTE_INT', 'LOTE_PROV', 'SERIE_INT', 'SERIE_PROV',
       'NUM_PALET', 'ALMACEN', 'ZONA_ORIGEN', 'SITUACION_ORIGEN',
       'UBICACION_ORIGEN', 'ZONA_DESTINO', 'SITUACION_DESTINO',
       'UBICACION_DESTINO', 'CANT_PRES', 'PRESENTACION', 'CAD_LOGISTICA',
       'UNIDADES1', 'UNIDADES2', 'ZONA_TRABAJO', 'MOTIVO_PARADA',
       'CONTADOR_ORIGEN', 'NUM_ETI_ENT', 'EJERCICIO_PEDIDO', 'SERIE_PEDIDO',
       'NUMERO_PEDIDO', 'COD_MVTO_COMPRA', 'COD_MVTO_COLOCA',
       'UNIDADES1_DEVOL', 'UNIDADES2_DEVOL', 'UBICACION_CARGA', 'CODIGO_MENU',
       'BOTON_CAMBIO_UBICACION', 'MAQUINA_ANT', 'ORGANIZACION_COMPRAS',
       'CODIGO_ORG_PLANTA', 'ORDEN_DE_FABRICACION', 'MOTIVO_BLOQUEO',
       'STATUS_ANT

```SQL
SELECT *
FROM RF_TAREAS_04
WHERE FECHA_GEN>=TO_DATE('05/05/2026','DD/MM/YYYY')
```

In [18]:
#@title Surtido DataFrame
Surtido = gc.open(working_file).worksheet('Surtido')

rows = Surtido.get_all_values()
Surtido_df = pd.DataFrame.from_records(rows[1:], columns=rows[0])
Surtido_df.columns

Index(['EMPRESA', 'CONTADOR', 'FECHA_GEN', 'USUARIO_GEN', 'MAQUINA_GEN',
       'FECHA_INICIO', 'FECHA_FIN', 'MAQUINA', 'USUARIO', 'OBSERVACIONES',
       'PRIORIDAD', 'STATUS', 'ALMACEN', 'ENTIDAD', 'ORG_COMERCIAL',
       'EJERCICIO_PEDIDO', 'SERIE_PEDIDO', 'NUMERO_PEDIDO', 'PROP_EJER',
       'PROP_NUMERO', 'ZONA_TRABAJO', 'SUB_TAREA', 'BOX', 'FECHA_PEDIDO',
       'FECHA_ENTREGA', 'FECHA_MODIFICACION', 'MOTIVO_PARADA',
       'CONTADOR_ORIGEN', 'TEXTO', 'SERIE_HP', 'ESPERAR_A', 'MOTIVO_BLOQUEO',
       'VOLUMEN', 'PESO', 'MULTITAREA', 'ID_MULTITAREA', 'SERIE_HC', 'ORIGEN',
       'TAREA_08', 'TIPO_PREPARACION', 'MAQUINA_ANT', 'ZONA_ALMACEN',
       'NIVEL_AGRUPACION', 'DOMICILIO_ENVIO', 'NUMERO_PEDIDO_CLIENTE',
       'AGENCIA_ENVIO', 'MOSTRAR_PEDIDO_UNICO', 'STATUS_ANTERIOR',
       'MOVIMIENTO_PREPARACION', 'MOVIMIENTO_ENVIO_BOX',
       'DOCUMENTO_PREPARACION', 'DOCUMENTO_ENVIO_BOX', 'ID_AGRUPACION',
       'TRABAJADOR', 'ASIGNACION_MAQUINA', 'ROTURA', 'FECHA_ENVIO_EXT',
       

```SQL
SELECT *
FROM RF_TAREAS_08
WHERE FECHA_GEN>=TO_DATE('05/05/2026','DD/MM/YYYY')
```

In [19]:
#@title Embarque DataFrame
Embarque = gc.open(working_file).worksheet('Embarque')

rows = Embarque.get_all_values()

Embarque_df = pd.DataFrame.from_records(rows[1:], columns=rows[0])
Embarque_df.columns

Index(['TAREA', 'EMPRESA', 'ORG_COMERCIAL', 'EJERCICIO_PEDIDO', 'SERIE_PEDIDO',
       'NUMERO_PEDIDO', 'AGENCIA', 'ALMACEN', 'BOX_PREPARACION',
       'ZONA_ALMACEN', 'UBICACION', 'ZONA_TRABAJO', 'FECHA_GEN', 'USUARIO_GEN',
       'MAQUINA_GEN', 'FECHA_INICIO', 'FECHA_FIN', 'PRIORIDAD', 'MAQUINA',
       'USUARIO', 'ESTATUS_TAREA', 'MOTIVO_BLOQUEO', 'NUMERO_PROP',
       'EJERCICIO_PROP', 'SERIE_HC', 'ORIGEN', 'MAQUINA_ANT',
       'NIVEL_AGRUPACION', 'DOMICILIO_ENVIO', 'NUMERO_PEDIDO_CLIENTE',
       'ENTIDAD', 'MOSTRAR_PEDIDO_UNICO', 'MOVIMIENTO_EMBALADO',
       'DOCUMENTO_EMBALADO', 'ID_AGRUPACION', 'SERIE_HP', 'BOX_EXPEDICION',
       'TRABAJADOR'],
      dtype='object')

In [20]:
import os

# List of DataFrames to export
dataframes = {
    'citas_df': citas_df,
    'Embarque_df': Embarque_df,
    'entradas_df': entradas_df,
    'lineas_df': lineas_df,
    'pedidos_df': pedidos_df,
    'Surtido_df': Surtido_df,
    'Ubicaciones_df': Ubicaciones_df
}

# Ensure the working folder exists
if not os.path.exists(working_folder):
    os.makedirs(working_folder)

# Export each DataFrame to CSV
for name, df in dataframes.items():
    file_path = os.path.join(working_folder, f'{name}.csv')
    df.to_csv(file_path, index=False)
    print(f'Exported: {file_path}')

Exported: /content/drive/MyDrive/Darovi Consulting/Zeus/reextraccindedatosproyectohades/citas_df.csv
Exported: /content/drive/MyDrive/Darovi Consulting/Zeus/reextraccindedatosproyectohades/Embarque_df.csv
Exported: /content/drive/MyDrive/Darovi Consulting/Zeus/reextraccindedatosproyectohades/entradas_df.csv
Exported: /content/drive/MyDrive/Darovi Consulting/Zeus/reextraccindedatosproyectohades/lineas_df.csv
Exported: /content/drive/MyDrive/Darovi Consulting/Zeus/reextraccindedatosproyectohades/pedidos_df.csv
Exported: /content/drive/MyDrive/Darovi Consulting/Zeus/reextraccindedatosproyectohades/Surtido_df.csv
Exported: /content/drive/MyDrive/Darovi Consulting/Zeus/reextraccindedatosproyectohades/Ubicaciones_df.csv
